# 02 · Modelos, alternativas y decisiones de desarrollo

Partimos de reglas sencillas y proponemos modelos más flexibles. Cada bloque presenta una hipótesis,
su contraste y la decisión siguiente. Las tablas incluyen alternativas descartadas y los resultados
del desarrollo, del contraste temporal y de la muestra acumulada.

**Captura del estudio:** 20 de octubre de 2026, temporada **2026/27 hasta J9**: **88 partidos
finalizados**. El desarrollo utiliza los **59 de las jornadas 1–6**; los **29 restantes** forman
el contraste temporal. El análisis es retrospectivo: este contraste no se presenta como un test
ciego independiente de toda la exploración. Los recuentos reales se muestran al cargar los datos.

**Distribución pública.** Se conservan las explicaciones y los resultados agregados del estudio. Los CSV reales y las salidas de partidos individuales no se incluyen. Para ejecutar las celdas, prepara primero tus datos en `data/local/actualizados` siguiendo [docs/guia_de_uso.md](../docs/guia_de_uso.md). Obtener otra captura no garantiza las mismas métricas.


In [1]:
from pathlib import Path
import os, sys
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")
RAIZ = Path.cwd() if (Path.cwd()/"pyproject.toml").exists() else Path.cwd().parent
assert (RAIZ/"src/faltas").is_dir(), "Abre el notebook desde la carpeta del proyecto o notebooks/."
sys.path.insert(0, str(RAIZ/"src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
from faltas.util import leer_json
from faltas.datos import datos_por_defecto, cargar_datos
from faltas.investigacion.biblioteca import tabla, mostrar, imagen, resumen_hipotesis
from faltas.entrega import cobertura, INFORMES_ACTUALES
get_ipython().run_line_magic("matplotlib", "inline")
CARPETA_ACTUALIZADA = RAIZ/"data/local/actualizados"
DATOS = (datos_por_defecto() if os.environ.get("FALTAS_DATOS") else
         CARPETA_ACTUALIZADA if CARPETA_ACTUALIZADA.exists() else datos_por_defecto())
d = cargar_datos(DATOS)
INFO_DATOS = cobertura(d)
print("Carpeta de datos elegida:", DATOS.relative_to(RAIZ) if DATOS.is_relative_to(RAIZ) else DATOS)
print("Captura:", INFO_DATOS["captura_utc"], "| Hasta jornada:", INFO_DATOS["jornada_hasta"])
print("Partidos actuales:", INFO_DATOS["partidos_actuales"], "| Historial finalizado:", INFO_DATOS["partidos_finalizados"])
INFORMES = RAIZ/"reports/investigacion"
if d.huella != leer_json(RAIZ/"config/diseno_estudio.json")["huella_datos"]:
    print("Los CSV elegidos difieren de la captura del estudio: las tablas archivadas siguen siendo V1 hasta J9.")

Carpeta de datos elegida: data\local\estudio_v1
Captura: 2026-10-20T10:36:55Z | Hasta jornada: 9
Partidos actuales: 88 | Historial finalizado: 1228


<a id="enfoques"></a>
## Tres estrategias que terminaremos comparando

El **global** comparte reglas entre todos los clubes; la **pareja** aprende con los dos solicitados
frente a todos sus rivales anteriores; la **referencia** promedia faltas propias y recibidas por el rival.
La pareja no usa únicamente enfrentamientos directos y el global no guarda una cifra fija por equipo.
Los tamaños 93 y 29 se justifican con las pruebas de bloques siguientes.

<a id="walk-forward"></a>
## Cómo entrenamos y evaluamos: walk-forward

1. Ajustar con los resultados disponibles antes del corte, de la temporada activa y las tres anteriores.
2. Predecir ambos equipos sin consultar la respuesta del encuentro; imputación y categorías también se ajustan dentro del entrenamiento.
3. Medir el error y avanzar; ese resultado podrá entrar en ajustes posteriores cuando esté disponible.

La selección compara los 59 partidos del tramo inicial. Los 29 posteriores se muestran como contraste
retrospectivo y los 88, como resumen acumulado. Fuera del entrenamiento no equivale a independiente de
la selección: probar y revisar muchas alternativas puede producir optimismo.

In [2]:
from faltas.investigacion.recorrido import preparar_contexto, evaluar_etapa, perfiles
ctx=preparar_contexto(DATOS)
REPETIR_ETAPAS=set()  # Opcional: {"inicial","ajuste","reducidos","contextuales","equilibrio","rejilla","confirmacion"}
catalogo=tabla("desarrollo","catalogo_38.csv")
mostrar("38 variantes: hipótesis, configuración, métricas y decisión",catalogo)
assert catalogo.modelo.nunique()==38
mostrar("Variantes nuevas por etapa",catalogo.groupby("primera_etapa",sort=False).size().rename("variantes").reset_index())

def comparar(etapa,controles=()):
    propias=tabla(etapa,"ranking.csv")
    comunes=tabla("desarrollo","ranking.csv")
    t=pd.concat([propias,comunes.loc[comunes.modelo.isin(controles)]],ignore_index=True).drop_duplicates("modelo")
    return t.sort_values(["RMSE_desarrollo","MAE_desarrollo","modelo"],kind="stable")

def repetir_si_se_pide(etapa):
    if etapa in REPETIR_ETAPAS:
        periodo=ctx["anterior"] if etapa=="confirmacion" else ctx["actual"]
        r=evaluar_etapa(ctx,etapa,periodo,n_jobs=2)
        mostrar("Recálculo sobre los CSV elegidos · "+etapa,r["metricas"])
    else:
        print("Se muestran los resultados completos guardados de V1 hasta J9. Repetición desactivada:",etapa)

modelo,primera_etapa,hipotesis_retrospectiva,hipotesis,perfil,receta,politica_filas,decision,partidos_desarrollo,n_desarrollo,MAE_desarrollo,RMSE_desarrollo,sesgo_desarrollo,partidos_contraste,n_contraste,MAE_contraste,RMSE_contraste,sesgo_contraste,partidos_acumulado,n_acumulado,MAE_acumulado,RMSE_acumulado,sesgo_acumulado
media_global_uniforme,inicial,False,Ignorar club/rival y recencia: nivel mínimo de comparación.,medias previas,{},actualidad_base,Alternativa documentada; no adoptada como receta operativa. Consultar el ranking de desarrollo de su etapa.,59,118,3.23373,3.97908,0.265414,29,58,3.0656,3.80119,0.424405,88,176,3.17832,3.92135,0.317809
media_global_recencia,inicial,False,Aislar recencia frente a una media uniforme.,medias previas,{},actualidad_base,Alternativa documentada; no adoptada como receta operativa. Consultar el ranking de desarrollo de su etapa.,59,118,3.25687,3.98411,0.38684,29,58,3.05162,3.7887,0.230295,88,176,3.18923,3.92079,0.335251
liga_localia_previa,inicial,False,Separar la referencia por localía.,medias previas,{},actualidad_base,Alternativa documentada; no adoptada como receta operativa. Consultar el ranking de desarrollo de su etapa.,59,118,3.23701,3.99055,0.38684,29,58,3.06193,3.79855,0.230295,88,176,3.17931,3.92832,0.335251
equipo_rival_previo,inicial,False,"Combinar al 50/50 faltas propias y recibidas del rival, con sus historiales previos.",medias previas,{},actualidad_base,Referencia operativa y control.,59,118,3.03895,3.84346,0.396573,29,58,2.78058,3.58096,0.256943,88,176,2.9538,3.75898,0.350558
poisson_glm,inicial,False,Candidato inicial reutilizado como control.,108 previas,"{""escalar"": true, ""familia"": ""poisson"", ""parametros"": {""alpha"": 1.0, ""fit_intercept"": true, ""max_iter"": 1000, ""solver"": ""lbfgs"", ""tol"": 1e-06, ""warm_start"": false}}",actualidad_base,Alternativa documentada; no adoptada como receta operativa. Consultar el ranking de desarrollo de su etapa.,59,118,3.5371,4.50339,0.151354,29,58,2.82542,3.70358,0.126546,88,176,3.30257,4.25645,0.143178
xgb_poisson,inicial,False,Candidato inicial reutilizado como control.,108 previas,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 3, ""min_child_weight"": 10, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""count:poisson"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 10.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}",actualidad_base,Alternativa documentada; no adoptada como receta operativa. Consultar el ranking de desarrollo de su etapa.,59,118,3.02354,3.7883,0.116025,29,58,2.83952,3.63102,0.0471991,88,176,2.96289,3.7372,0.0933434
xgb_cuadratico,inicial,False,Candidato inicial reutilizado como control.,108 previas,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.0, ""max_depth"": 3, ""min_child_weight"": 10, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""reg:squarederror"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 10.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}",actualidad_base,Alternativa documentada; no adoptada como receta operativa. Consultar el ranking de desarrollo de su etapa.,59,118,3.05965,3.81146,0.234712,29,58,2.82752,3.70219,0.2338,88,176,2.98315,3.7758,0.234411
xgb_poisson_sin_grupos,ajuste,False,Ablación de grupos y todas las variables vs_grupo; mismos cortes e IDs.,93 sin grupos,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 3, ""min_child_weight"": 10, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""count:poisson"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 10.0,

primera_etapa,variantes
inicial,7
ajuste,8
reducidos,12
contextuales,10
equilibrio,1


<a id="inicial"></a>
## 1. Reglas sencillas y candidatos generales

Cuatro referencias comparan media uniforme, recencia, liga por localía y equipo/rival.
Se enfrentan a Poisson y dos XGBoost con distinta pérdida, inicialmente con 108 entradas.
Los métodos ponderados parten de 55/25/13/7 por temporada, más recencia interna; la media uniforme
actúa como control. La pregunta es si aprender reglas complejas mejora esas referencias.

In [3]:
mostrar("Comparación inicial: desarrollo, contraste y acumulado",comparar("inicial"),abierto=True)
if (INFORMES / "inicial" / "metricas_cortes.csv").is_file():
    mostrar("Errores por corte", tabla("inicial", "metricas_cortes.csv"))
else:
    print("Los errores por corte individual no se distribuyen. "
          "Se conservan las métricas agregadas y el código para obtener los detalles localmente.")
mostrar("Entrenamiento, convergencia y avisos",tabla("inicial","diagnostico_ajuste.csv"))
mostrar("Masas reales por temporada",tabla("inicial","resumen_masas.csv"))
repetir_si_se_pide("inicial")

modelo,partidos_desarrollo,n_desarrollo,MAE_desarrollo,RMSE_desarrollo,sesgo_desarrollo,partidos_contraste,n_contraste,MAE_contraste,RMSE_contraste,sesgo_contraste,partidos_acumulado,n_acumulado,MAE_acumulado,RMSE_acumulado,sesgo_acumulado,hipotesis_retrospectiva
xgb_poisson,59,118,3.02354,3.7883,0.116025,29,58,2.83952,3.63102,0.0471991,88,176,2.96289,3.7372,0.0933434,False
xgb_cuadratico,59,118,3.05965,3.81146,0.234712,29,58,2.82752,3.70219,0.2338,88,176,2.98315,3.7758,0.234411,False
equipo_rival_previo,59,118,3.03895,3.84346,0.396573,29,58,2.78058,3.58096,0.256943,88,176,2.9538,3.75898,0.350558,False
media_global_uniforme,59,118,3.23373,3.97908,0.265414,29,58,3.0656,3.80119,0.424405,88,176,3.17832,3.92135,0.317809,False
media_global_recencia,59,118,3.25687,3.98411,0.38684,29,58,3.05162,3.7887,0.230295,88,176,3.18923,3.92079,0.335251,False
liga_localia_previa,59,118,3.23701,3.99055,0.38684,29,58,3.06193,3.79855,0.230295,88,176,3.17931,3.92832,0.335251,False
poisson_glm,59,118,3.5371,4.50339,0.151354,29,58,2.82542,3.70358,0.126546,88,176,3.30257,4.25645,0.143178,False


Los errores por corte individual no se distribuyen. Se conservan las métricas agregadas y el código para obtener los detalles localmente.


modelo,observaciones_train,partidos_train,observaciones_valid,variables_salida,n_columnas_valor_excluido,peso_medio,peso_minimo,peso_maximo,RMSE_train,RMSE_train_ponderado,MAE_train,n_iter,correcciones_dominio_valid,segundos_ajuste_y_prediccion,ajustes,ajustes_con_aviso
poisson_glm,2361.6,1180.8,2.07059,283.624,6,1,0.0590563,28.3825,3.78353,2.93476,2.95674,146.529,0,1.0345,85,0
xgb_poisson,2361.6,1180.8,2.07059,283.624,6,1,0.0590563,28.3825,3.6595,2.79817,2.88019,100,0,2.12446,85,0
xgb_cuadratico,2361.6,1180.8,2.07059,283.624,6,1,0.0590563,28.3825,3.54457,2.53431,2.77328,100,0,2.05126,85,0


temporada,partidos_mean,partidos_min,partidos_max,observaciones_mean,observaciones_min,observaciones_max,peso_total_mean,peso_total_min,peso_total_max,peso_medio_mean,peso_medio_min,peso_medio_max,porcentaje_peso_mean,porcentaje_peso_min,porcentaje_peso_max
2025/26,380,380,380,760,760,760,623.184,572,1266.67,0.819979,0.752632,1.66667,26.4379,25,55.5556
2024/25,380,380,380,760,760,760,324.056,297.44,658.667,0.426389,0.391368,0.866667,13.7477,13,28.8889
2023/24,380,380,380,760,760,760,174.492,160.16,354.667,0.229594,0.210737,0.466667,7.40261,7,15.5556
2026/27,42.8148,4,86,85.6296,8,172,1301.1,1258.4,1348.6,27.9071,7.8407,157.3,55,55,55


Se muestran los resultados completos guardados de V1 hasta J9. Repetición desactivada: inicial


**Decisión.** Las diferencias no justifican asumir que ML siempre mejora una media.
Se mantienen referencias como controles y se comprueba si regularizar o retirar variables reduce
el error. La comparación prioriza RMSE del desarrollo y usa MAE para desempatar.

<a id="ajuste"></a>
## 2. Regularización y ablación de grupos

Se comparan tres candidatos iniciales y ocho variantes de complejidad, pesos de filas y grupos,
junto a las cuatro referencias. También se prueban pesos 65/22/9/4 y 51/29/14/6.
Retirar grupos deja 93 entradas. Para aislar cambios de pesos
de entrenamiento se mantienen las medias previas de las entradas. Se examinan también convergencia,
errores por segmentos y concentración de los fallos; estos diagnósticos no eliminan casos difíciles.

In [4]:
mostrar("Recetas y sus hipótesis",resumen_hipotesis("ajuste"))
mostrar("Ranking con controles",comparar("ajuste",["media_global_uniforme","media_global_recencia","liga_localia_previa","equipo_rival_previo","poisson_glm","xgb_cuadratico","xgb_poisson"]),abierto=True)
mostrar("Convergencia y error de entrenamiento",tabla("ajuste","diagnostico_ajuste.csv"))
mostrar("Localía, club, contexto, soporte y tramos",tabla("ajuste","segmentos.csv"))
mostrar("Concentración en los cinco errores mayores",tabla("ajuste","concentracion.csv"))
mostrar("Comparación pareada en desarrollo",pd.DataFrame([leer_json(INFORMES/"desarrollo/intervalos.json")["ajuste"]]))
repetir_si_se_pide("ajuste")

variante,ambito,perfil,hipotesis,pesos_filas,receta
poisson_glm,global,108 previas,Candidato inicial reutilizado como control.,actualidad_base,"{""escalar"": true, ""familia"": ""poisson"", ""parametros"": {""alpha"": 1.0, ""fit_intercept"": true, ""max_iter"": 1000, ""solver"": ""lbfgs"", ""tol"": 1e-06, ""warm_start"": false}}"
xgb_poisson,global,108 previas,Candidato inicial reutilizado como control.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 3, ""min_child_weight"": 10, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""count:poisson"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 10.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}"
xgb_cuadratico,global,108 previas,Candidato inicial reutilizado como control.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.0, ""max_depth"": 3, ""min_child_weight"": 10, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""reg:squarederror"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 10.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}"
poisson_alpha10,global,108 previas,Reducir varianza mediante mayor regularización.,actualidad_base,"{""escalar"": true, ""familia"": ""poisson"", ""parametros"": {""alpha"": 10.0, ""fit_intercept"": true, ""max_iter"": 1000, ""solver"": ""lbfgs"", ""tol"": 1e-06, ""warm_start"": false}}"
poisson_alpha100,global,108 previas,Contrastar una contracción más intensa.,actualidad_base,"{""escalar"": true, ""familia"": ""poisson"", ""parametros"": {""alpha"": 100.0, ""fit_intercept"": true, ""max_iter"": 1000, ""solver"": ""lbfgs"", ""tol"": 1e-06, ""warm_start"": false}}"
xgb_poisson_suave,global,108 previas,Árboles más simples y mayor regularización.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 2, ""min_child_weight"": 20, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""count:poisson"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 20.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}"
xgb_poisson_200,global,108 previas,"Más pasos de aprendizaje con mayor regularización; bloque conjunto, no atribución aislada.",actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 3, ""min_child_weight"": 20, ""n_estimators"": 200, ""n_jobs"": 1, ""objective"": ""count:poisson"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 20.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}"
xgb_cuadratico_suave,global,108 previas,Comprobar si el objetivo cuadrático mejora al simplificar.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.0, ""max_depth"": 2, ""min_child_weight"": 20, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""reg:squarederror"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 20.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}"
xgb_poisson_marcada,global,108 previas,"Aislar mayor peso de actualidad en la pérdida, con X fijo.",actualidad_marcada,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 3, ""min_child_weight"": 10, ""n_estimators"": 

modelo,partidos_desarrollo,n_desarrollo,MAE_desarrollo,RMSE_desarrollo,sesgo_desarrollo,partidos_contraste,n_contraste,MAE_contraste,RMSE_contraste,sesgo_contraste,partidos_acumulado,n_acumulado,MAE_acumulado,RMSE_acumulado,sesgo_acumulado,hipotesis_retrospectiva
xgb_poisson_sin_grupos,59,118,2.99863,3.75474,0.156173,29,58,2.90109,3.73584,0.139625,88,176,2.96648,3.74852,0.15072,False
xgb_poisson_moderada,59,118,3.0075,3.76976,0.107277,29,58,2.81738,3.66721,0.083083,88,176,2.94485,3.73628,0.0993039,False
xgb_poisson,59,118,3.02354,3.7883,0.116025,29,58,2.83952,3.63102,0.0471991,88,176,2.96289,3.7372,0.0933434,False
xgb_cuadratico,59,118,3.05965,3.81146,0.234712,29,58,2.82752,3.70219,0.2338,88,176,2.98315,3.7758,0.234411,False
xgb_poisson_200,59,118,3.06538,3.8215,0.0549021,29,58,2.85569,3.67497,-0.0117477,88,176,2.99628,3.77384,0.032938,False
xgb_poisson_suave,59,118,3.04151,3.83728,0.109088,29,58,2.77429,3.62661,0.100703,88,176,2.95345,3.76916,0.106325,False
poisson_alpha100,59,118,3.07601,3.83867,0.335744,29,58,2.76908,3.60108,0.0660498,88,176,2.97486,3.76203,0.246868,False
equipo_rival_previo,59,118,3.03895,3.84346,0.396573,29,58,2.78058,3.58096,0.256943,88,176,2.9538,3.75898,0.350558,False
xgb_cuadratico_suave,59,118,3.11207,3.85976,0.136837,29,58,2.85623,3.68018,0.127988,88,176,3.02776,3.80152,0.133921,False
xgb_poisson_marcada,59,118,3.07749,3.88559,0.0221124,29,58,2.80334,3.63306,0.060053,88,176,2.98715,3.80422,0.0346156,False


modelo,observaciones_train,partidos_train,observaciones_valid,variables_salida,n_columnas_valor_excluido,peso_medio,peso_minimo,peso_maximo,RMSE_train,RMSE_train_ponderado,MAE_train,n_iter,correcciones_dominio_valid,segundos_ajuste_y_prediccion,ajustes,ajustes_con_aviso
poisson_alpha10,2361.6,1180.8,2.07059,283.624,6,1,0.0590563,28.3825,3.76974,3.19437,2.9679,45.4824,0,0.820141,85,0
poisson_alpha100,2361.6,1180.8,2.07059,283.624,6,1,0.0590563,28.3825,3.85554,3.52791,3.04911,17.9412,0,0.757803,85,0
xgb_poisson_suave,2361.6,1180.8,2.07059,283.624,6,1,0.0590563,28.3825,3.76384,3.10006,2.975,100,0,1.86631,85,0
xgb_poisson_200,2361.6,1180.8,2.07059,283.624,6,1,0.0590563,28.3825,3.53159,2.51179,2.7657,200,0,3.21176,85,0
xgb_cuadratico_suave,2361.6,1180.8,2.07059,283.624,6,1,0.0590563,28.3825,3.67954,2.87803,2.89522,100,0,1.78012,85,0
xgb_poisson_marcada,2361.6,1180.8,2.07059,283.624,6,1,0.0152984,34.6167,3.7024,2.5748,2.91365,100,0,2.16066,85,0
xgb_poisson_moderada,2361.6,1180.8,2.07059,283.624,6,1,0.0732798,25.9274,3.65111,2.86484,2.87314,100,0,2.18701,85,0
xgb_poisson_sin_grupos,2361.6,1180.8,2.07059,238.624,6,1,0.0590563,28.3825,3.67393,2.80162,2.88879,100,0,1.86781,85,0


modelo,segmento,valor,n,partidos,RMSE,MAE,sesgo,muestra_pequena
poisson_alpha10,localia,Local,88,88,4.03145,3.25351,0.191664,False
poisson_alpha10,localia,Visitante,88,88,3.81527,2.94714,0.151069,False
poisson_alpha100,localia,Local,88,88,3.87376,3.15554,0.148882,False
poisson_alpha100,localia,Visitante,88,88,3.64688,2.79418,0.344854,False
xgb_poisson_suave,localia,Local,88,88,3.92281,3.20237,-0.00650313,False
xgb_poisson_suave,localia,Visitante,88,88,3.60897,2.70454,0.219153,False
xgb_poisson_200,localia,Local,88,88,3.86765,3.17322,-0.0784426,False
xgb_poisson_200,localia,Visitante,88,88,3.67764,2.81934,0.144319,False
xgb_cuadratico_suave,localia,Local,88,88,3.96261,3.22906,0.0384428,False
xgb_cuadratico_suave,localia,Visitante,88,88,3.63329,2.82646,0.229399,False


modelo,fraccion_error_cuadrado_5_peores
poisson_alpha10,0.197881
poisson_alpha100,0.197814
xgb_poisson_suave,0.201903
xgb_poisson_200,0.210464
xgb_cuadratico_suave,0.198082
xgb_poisson_marcada,0.212586
xgb_poisson_moderada,0.215914
xgb_poisson_sin_grupos,0.208709


modelo,referencia,partidos,delta_RMSE,p025_descriptivo,p975_descriptivo,limite
xgb_poisson_sin_grupos,equipo_rival_previo,59,-0.0887192,-0.313021,0.141445,"Intervalo exploratorio por partidos, condicionado a datos y selección; no demuestra superioridad ni corrige búsqueda ni dependencia temporal."


Se muestran los resultados completos guardados de V1 hasta J9. Repetición desactivada: ajuste


**Decisión.** Continúa XGBoost Poisson con 93 entradas como candidato global. La mejora
en desarrollo es pequeña y su intervalo exploratorio incluye cero. No demuestra que los grupos
carezcan de utilidad en cualquier modelo; tampoco garantiza que este candidato lidere todos
los tramos posteriores. Las columnas de contraste y acumulado permiten comprobarlo.

<a id="reducidos"></a>
## 3. Menos entradas y aprendizaje por pareja

Doce variantes prueban núcleos de 7 y 13 entradas, selección interna de cinco y estrategias por pareja.
La selección utiliza una validación cronológica dentro del entrenamiento exterior. Se conserva una
hipótesis manual de grupos, identificada como retrospectiva y excluida de la comparación temporal
estricta. Se pregunta si simplificar o especializar el ajuste mejora al global de 93.

In [5]:
mostrar("Doce variantes reducidas",resumen_hipotesis("reducidos"))
pp=perfiles(ctx)
mostrar("Entradas exactas de los perfiles pequeños",pd.DataFrame([dict(perfil=n,numero=len(x.columns),entradas=" | ".join(x.columns)) for n,x in pp["reducidos"].items()]))
mostrar("Ranking reducido con controles",comparar("reducidos",["xgb_poisson_sin_grupos","equipo_rival_previo"]))
mostrar("Errores pequeños, grandes y dispersión",tabla("reducidos","precision.csv"))
mostrar("Selección e importancia interna",tabla("reducidos","resumen_importancias.csv"))
repetir_si_se_pide("reducidos")

variante,ambito,perfil,hipotesis,pesos_filas,receta
general_nucleo7_poisson,general,nucleo7,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": true, ""familia"": ""poisson"", ""parametros"": {""alpha"": 1.0, ""fit_intercept"": true, ""max_iter"": 1000, ""solver"": ""lbfgs"", ""tol"": 1e-06, ""warm_start"": false}}"
general_nucleo7_xgboost,general,nucleo7,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 1.0, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 2, ""min_child_weight"": 5, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""count:poisson"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 5.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}"
general_contexto13_poisson,general,contexto13,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": true, ""familia"": ""poisson"", ""parametros"": {""alpha"": 1.0, ""fit_intercept"": true, ""max_iter"": 1000, ""solver"": ""lbfgs"", ""tol"": 1e-06, ""warm_start"": false}}"
general_contexto13_xgboost,general,contexto13,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 1.0, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 2, ""min_child_weight"": 5, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""count:poisson"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 5.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}"
pareja_nucleo7_poisson,pareja,nucleo7,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": true, ""familia"": ""poisson"", ""parametros"": {""alpha"": 1.0, ""fit_intercept"": true, ""max_iter"": 1000, ""solver"": ""lbfgs"", ""tol"": 1e-06, ""warm_start"": false}}"
pareja_nucleo7_xgboost,pareja,nucleo7,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 1.0, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 2, ""min_child_weight"": 5, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""count:poisson"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 5.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}"
pareja_contexto13_poisson,pareja,contexto13,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": true, ""familia"": ""poisson"", ""parametros"": {""alpha"": 1.0, ""fit_intercept"": true, ""max_iter"": 1000, ""solver"": ""lbfgs"", ""tol"": 1e-06, ""warm_start"": false}}"
pareja_contexto13_xgboost,pareja,contexto13,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 1.0, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 2, ""min_child_weight"": 5, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""count:poisson"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 5.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}"
general_seleccion5,general,contexto13,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 1.0, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 2, ""min_child_weight"": 5, ""n_estimators"": 100, ""n_jobs"": 1,

perfil,numero,entradas
nucleo7,7,es_local | cometidas_actuales | recibidas_rival_actuales | cometidas_ultimos5 | recibidas_rival_ultimos5 | cometidas_referencia | recibidas_rival_referencia
contexto13,13,es_local | cometidas_actuales | recibidas_rival_actuales | cometidas_ultimos5 | recibidas_rival_ultimos5 | cometidas_referencia | recibidas_rival_referencia | cruce_grupos | diferencia_posesion | cometidas_similares | recibidas_rival_similares | n_similares_equipo | n_similares_rival
manual13,13,es_local | cometidas_actuales | recibidas_rival_actuales | cometidas_ultimos5 | recibidas_rival_ultimos5 | cometidas_referencia | recibidas_rival_referencia | cruce_grupos | diferencia_posesion | cometidas_similares | recibidas_rival_similares | n_similares_equipo | n_similares_rival


modelo,partidos_desarrollo,n_desarrollo,MAE_desarrollo,RMSE_desarrollo,sesgo_desarrollo,partidos_contraste,n_contraste,MAE_contraste,RMSE_contraste,sesgo_contraste,partidos_acumulado,n_acumulado,MAE_acumulado,RMSE_acumulado,sesgo_acumulado,hipotesis_retrospectiva
xgb_poisson_sin_grupos,59,118,2.99863,3.75474,0.156173,29,58,2.90109,3.73584,0.139625,88,176,2.96648,3.74852,0.15072,False
pareja_analogos,59,118,3.08421,3.79837,0.340618,29,58,3.02766,3.86906,0.260442,88,176,3.06558,3.82181,0.314197,False
general_nucleo7_poisson,59,118,3.07774,3.81367,0.224357,29,58,2.8163,3.6442,0.0558468,88,176,2.99159,3.75867,0.168825,False
general_nucleo7_xgboost,59,118,3.07074,3.838,0.244314,29,58,2.8796,3.6918,0.0578179,88,176,3.00775,3.79044,0.182855,False
equipo_rival_previo,59,118,3.03895,3.84346,0.396573,29,58,2.78058,3.58096,0.256943,88,176,2.9538,3.75898,0.350558,False
pareja_nucleo7_xgboost,59,118,3.04371,3.85863,0.348775,29,58,3.06339,3.82965,0.043411,88,176,3.0502,3.8491,0.248143,False
general_contexto13_poisson,59,118,3.1308,3.88047,0.0384989,29,58,2.81173,3.6194,0.0105121,88,176,3.02565,3.79642,0.029276,False
general_contexto13_xgboost,59,118,3.11728,3.92221,0.174298,29,58,2.90834,3.78242,0.0496544,88,176,3.04843,3.8767,0.133223,False
general_seleccion5,59,118,3.13916,3.94214,0.21337,29,58,2.84495,3.67037,-0.00211499,88,176,3.04221,3.85469,0.142358,False
pareja_contexto13_xgboost,59,118,3.09929,3.95434,0.286247,29,58,2.95447,3.77736,0.000556453,88,176,3.05157,3.89691,0.192099,False


modelo,partidos,n,error_hasta2_pct,error_hasta3_pct,error_desde5_pct,max_error_absoluto,correlacion,desv_prediccion,desv_observada
general_nucleo7_poisson,88,176,38.0682,57.3864,18.1818,11.1585,0.283651,1.34126,3.91958
general_nucleo7_xgboost,88,176,42.0455,57.3864,17.6136,11.6941,0.260339,1.32719,3.91958
general_contexto13_poisson,88,176,39.7727,56.25,16.4773,12.7016,0.261414,1.45017,3.91958
general_contexto13_xgboost,88,176,43.75,51.7045,18.75,12.8818,0.207284,1.44003,3.91958
pareja_nucleo7_poisson,88,176,40.3409,52.2727,22.1591,9.88029,0.269801,2.31421,3.91958
pareja_nucleo7_xgboost,88,176,41.4773,56.25,23.2955,10.4499,0.293746,2.04627,3.91958
pareja_contexto13_poisson,88,176,34.6591,45.4545,27.8409,12.4316,0.155602,2.65506,3.91958
pareja_contexto13_xgboost,88,176,41.4773,56.25,22.1591,13.6104,0.287216,2.19339,3.91958
general_seleccion5,88,176,42.0455,56.8182,18.75,12.5424,0.218729,1.40057,3.91958
general_manual13,88,176,38.6364,56.25,22.7273,11.3709,0.181575,1.49595,3.91958


campo,evaluaciones,veces_seleccionada,aumento_RMSE_medio,posicion_media,frecuencia_seleccion
cometidas_actuales,85,45,0.0251833,1,0.529412
cometidas_referencia,85,85,0.307541,5,1
cometidas_similares,85,53,0.0122347,9,0.623529
cometidas_ultimos5,85,12,-0.00099344,3,0.141176
cruce_grupos,85,18,0.00172045,7,0.211765
diferencia_posesion,85,15,-0.0015395,8,0.176471
es_local,85,7,-0.000137617,0,0.0823529
n_similares_equipo,85,4,-0.00610492,11,0.0470588
n_similares_rival,85,25,-0.00507958,12,0.294118
recibidas_rival_actuales,85,33,0.000666445,2,0.388235


Se muestran los resultados completos guardados de V1 hasta J9. Repetición desactivada: reducidos


**Decisión.** Los perfiles pequeños no mejoran el RMSE de desarrollo del candidato global.
Se descartan como receta operativa, manteniendo sus resultados. El ajuste por pareja se lleva
a perfiles más completos: producir predicciones distintas no acredita todavía mayor precisión.

<a id="contextuales"></a>
## 4. Contexto de equipo y rival

Diez variantes contrastan 18, 26, 28 y 32 características: localía, perfiles previos, grupos y
enfrentamientos directos. La pareja añade identidad del equipo objetivo. Se ponderan antecedentes
por afinidad de estilo sin conocer las faltas reales de la consulta, conservando todo el historial
elegible para no depender de uno o dos partidos muy similares.

In [6]:
from faltas.investigacion.metodos.contextuales import diccionario_contextual
mostrar("Diccionario contextual",diccionario_contextual())
mostrar("Diez variantes",resumen_hipotesis("contextuales"))
mostrar("Bloques reales de entrada",pd.DataFrame([dict(perfil=n,entradas=len(x.columns),columnas=" | ".join(x.columns)) for n,x in pp["contextuales"].items()]))
mostrar("Ranking con controles",comparar("contextuales",["xgb_poisson_sin_grupos","equipo_rival_previo"]))
mostrar("Precisión y dispersión",tabla("contextuales","precision.csv"))
mostrar("Segmentos",tabla("contextuales","segmentos.csv"))
repetir_si_se_pide("contextuales")

campo,origen,bloque,utilidad
es_local,es_local,perfil,"Localía, cruce K2 previo, posesión y puntos por partido de ambos equipos; dominio y rendimiento separados."
cruce_grupos,cruce_grupos,perfil,"Localía, cruce K2 previo, posesión y puntos por partido de ambos equipos; dominio y rendimiento separados."
posesion_equipo,equipo_actual_posesion,perfil,"Localía, cruce K2 previo, posesión y puntos por partido de ambos equipos; dominio y rendimiento separados."
posesion_rival,rival_actual_posesion,perfil,"Localía, cruce K2 previo, posesión y puntos por partido de ambos equipos; dominio y rendimiento separados."
puntos_equipo,equipo_actual_puntos,perfil,"Localía, cruce K2 previo, posesión y puntos por partido de ambos equipos; dominio y rendimiento separados."
puntos_rival,rival_actual_puntos,perfil,"Localía, cruce K2 previo, posesión y puntos por partido de ambos equipos; dominio y rendimiento separados."
cometidas_actuales,equipo_actual_cometidas,faltas,Faltas cometidas y recibidas actuales de ambos; forma reciente e historial ponderado en los dos papeles relevantes.
recibidas_rival_actuales,rival_actual_recibidas,faltas,Faltas cometidas y recibidas actuales de ambos; forma reciente e historial ponderado en los dos papeles relevantes.
cometidas_ultimos5,equipo_ultimos5_cometidas,faltas,Faltas cometidas y recibidas actuales de ambos; forma reciente e historial ponderado en los dos papeles relevantes.
recibidas_rival_ultimos5,rival_ultimos5_recibidas,faltas,Faltas cometidas y recibidas actuales de ambos; forma reciente e historial ponderado en los dos papeles relevantes.


variante,ambito,perfil,hipotesis,pesos_filas,receta
global_perfil_localia,general,perfil_localia,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 3, ""min_child_weight"": 10, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""count:poisson"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 10.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}"
global_perfil_grupos,general,perfil_grupos,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 3, ""min_child_weight"": 10, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""count:poisson"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 10.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}"
global_perfil_directos,general,perfil_directos,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 3, ""min_child_weight"": 10, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""count:poisson"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 10.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}"
global_perfil_analogos,general,perfil_analogos,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 3, ""min_child_weight"": 10, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""count:poisson"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 10.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}"
global_perfil_directos_poisson,general,perfil_directos,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": true, ""familia"": ""poisson"", ""parametros"": {""alpha"": 1.0, ""fit_intercept"": true, ""max_iter"": 1000, ""solver"": ""lbfgs"", ""tol"": 1e-06, ""warm_start"": false}}"
global_perfil_analogos_poisson,general,perfil_analogos,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": true, ""familia"": ""poisson"", ""parametros"": {""alpha"": 1.0, ""fit_intercept"": true, ""max_iter"": 1000, ""solver"": ""lbfgs"", ""tol"": 1e-06, ""warm_start"": false}}"
local_perfil_directos,pareja,perfil_directos,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 3, ""min_child_weight"": 10, ""n_estimators"": 100, ""n_jobs"": 1, ""objective"": ""count:poisson"", ""random_state"": 42, ""reg_alpha"": 0.0, ""reg_lambda"": 10.0, ""subsample"": 1.0, ""tree_method"": ""hist"", ""validate_parameters"": true, ""verbosity"": 0}}"
local_perfil_analogos,pareja,perfil_analogos,Contraste del bloque de variables y estrategia de entrenamiento indicados.,actualidad_base,"{""escalar"": false, ""familia"": ""xgboost"", ""parametros"": {""colsample_bytree"": 0.8, ""device"": ""cpu"", ""learning_rate"": 0.05, ""max_delta_step"": 0.7, ""max_depth"": 3, ""min_child_weight"": 10, ""n_estimators"": 100, ""n_jobs"": 1, ""objectiv

perfil,entradas,columnas
perfil_localia,18,es_local | cruce_grupos | posesion_equipo | posesion_rival | puntos_equipo | puntos_rival | cometidas_actuales | recibidas_rival_actuales | cometidas_ultimos5 | recibidas_rival_ultimos5 | cometidas_referencia | recibidas_rival_referencia | recibidas_equipo_actuales | cometidas_rival_actuales | cometidas_localia | recibidas_rival_localia | n_equipo_actual | n_rival_actual
perfil_grupos,26,es_local | cruce_grupos | posesion_equipo | posesion_rival | puntos_equipo | puntos_rival | cometidas_actuales | recibidas_rival_actuales | cometidas_ultimos5 | recibidas_rival_ultimos5 | cometidas_referencia | recibidas_rival_referencia | recibidas_equipo_actuales | cometidas_rival_actuales | cometidas_localia | recibidas_rival_localia | n_equipo_actual | n_rival_actual | cometidas_similares_actuales | recibidas_rival_similares_actuales | cometidas_similares_anteriores | recibidas_rival_similares_anteriores | n_similares_equipo_actual | n_similares_rival_actual | n_similares_equipo_anterior | n_similares_rival_anterior
perfil_directos,28,es_local | cruce_grupos | posesion_equipo | posesion_rival | puntos_equipo | puntos_rival | cometidas_actuales | recibidas_rival_actuales | cometidas_ultimos5 | recibidas_rival_ultimos5 | cometidas_referencia | recibidas_rival_referencia | recibidas_equipo_actuales | cometidas_rival_actuales | cometidas_localia | recibidas_rival_localia | n_equipo_actual | n_rival_actual | cometidas_similares_actuales | recibidas_rival_similares_actuales | cometidas_similares_anteriores | recibidas_rival_similares_anteriores | n_similares_equipo_actual | n_similares_rival_actual | n_similares_equipo_anterior | n_similares_rival_anterior | cometidas_directos_anteriores | n_directos_anteriores
perfil_analogos,32,es_local | cruce_grupos | posesion_equipo | posesion_rival | puntos_equipo | puntos_rival | cometidas_actuales | recibidas_rival_actuales | cometidas_ultimos5 | recibidas_rival_ultimos5 | cometidas_referencia | recibidas_rival_referencia | recibidas_equipo_actuales | cometidas_rival_actuales | cometidas_localia | recibidas_rival_localia | n_equipo_actual | n_rival_actual | cometidas_similares_actuales | recibidas_rival_similares_actuales | cometidas_similares_anteriores | recibidas_rival_similares_anteriores | n_similares_equipo_actual | n_similares_rival_actual | n_similares_equipo_anterior | n_similares_rival_anterior | cometidas_directos_anteriores | n_directos_anteriores | cometidas_analogos | recibidas_rival_analogos | n_efectivo_analogos | n_efectivo_rival_analogos


modelo,partidos_desarrollo,n_desarrollo,MAE_desarrollo,RMSE_desarrollo,sesgo_desarrollo,partidos_contraste,n_contraste,MAE_contraste,RMSE_contraste,sesgo_contraste,partidos_acumulado,n_acumulado,MAE_acumulado,RMSE_acumulado,sesgo_acumulado,hipotesis_retrospectiva
xgb_poisson_sin_grupos,59,118,2.99863,3.75474,0.156173,29,58,2.90109,3.73584,0.139625,88,176,2.96648,3.74852,0.15072,False
local_perfil_directos,59,118,3.02335,3.78079,0.330944,29,58,3.2923,4.20627,0.0896242,88,176,3.11198,3.9261,0.251418,False
local_analogos_simple,59,118,3.00552,3.81195,0.354763,29,58,3.09368,4.02674,0.294827,88,176,3.03457,3.88404,0.335011,False
local_perfil_analogos,59,118,3.06308,3.82477,0.339638,29,58,3.2287,4.08059,0.229908,88,176,3.11766,3.91092,0.303477,False
global_perfil_localia,59,118,3.02897,3.82744,0.275469,29,58,3.03264,3.884,0.0937509,88,176,3.03018,3.84617,0.215585,False
equipo_rival_previo,59,118,3.03895,3.84346,0.396573,29,58,2.78058,3.58096,0.256943,88,176,2.9538,3.75898,0.350558,False
global_perfil_directos,59,118,3.04522,3.88561,0.134642,29,58,2.93718,3.82146,-0.106353,88,176,3.00962,3.86459,0.0552232,False
global_perfil_analogos,59,118,3.05747,3.89946,0.149704,29,58,2.93839,3.80287,-0.0517177,88,176,3.01823,3.86789,0.0833266,False
global_perfil_grupos,59,118,3.09293,3.90112,0.129331,29,58,2.87505,3.7973,-0.0861698,88,176,3.02113,3.86721,0.0583136,False
global_perfil_directos_poisson,59,118,3.20623,4.07889,0.0149172,29,58,2.85753,3.69611,-0.0224101,88,176,3.09132,3.95684,0.00261615,False


modelo,partidos,n,error_hasta2_pct,error_hasta3_pct,error_desde5_pct,max_error_absoluto,correlacion,desv_prediccion,desv_observada
global_perfil_localia,88,176,41.4773,55.6818,21.5909,11.0993,0.235966,1.49334,3.91958
global_perfil_grupos,88,176,42.6136,53.4091,17.0455,13.448,0.224791,1.55235,3.91958
global_perfil_directos,88,176,41.4773,57.9545,18.1818,13.7748,0.221711,1.50886,3.91958
global_perfil_analogos,88,176,41.4773,55.6818,18.1818,13.9525,0.223279,1.53993,3.91958
global_perfil_directos_poisson,88,176,39.2045,57.3864,19.8864,13.5928,0.221889,1.93711,3.91958
global_perfil_analogos_poisson,88,176,42.0455,56.25,19.8864,13.8085,0.199691,1.95519,3.91958
local_perfil_directos,88,176,41.4773,56.25,23.2955,10.8062,0.283938,2.25933,3.91958
local_perfil_analogos,88,176,38.6364,57.9545,21.0227,10.8519,0.293781,2.27084,3.91958
local_analogos_simple,88,176,41.4773,57.3864,21.0227,11.4452,0.30882,2.28804,3.91958
local_analogos_poisson,88,176,32.3864,50,26.7045,13.2853,0.179068,3.04917,3.91958


modelo,segmento,valor,n,partidos,RMSE,MAE,sesgo,muestra_pequena
global_perfil_localia,localia,Local,88,88,3.93278,3.14948,0.0772803,False
global_perfil_localia,localia,Visitante,88,88,3.75757,2.91088,0.353889,False
global_perfil_grupos,localia,Local,88,88,3.93055,3.16528,-0.173882,False
global_perfil_grupos,localia,Visitante,88,88,3.80282,2.87698,0.290509,False
global_perfil_directos,localia,Local,88,88,3.91197,3.15813,-0.151005,False
global_perfil_directos,localia,Visitante,88,88,3.81661,2.8611,0.261451,False
global_perfil_analogos,localia,Local,88,88,3.92341,3.15231,-0.15651,False
global_perfil_analogos,localia,Visitante,88,88,3.81157,2.88414,0.323164,False
global_perfil_directos_poisson,localia,Local,88,88,4.08794,3.30621,-0.00535775,False
global_perfil_directos_poisson,localia,Visitante,88,88,3.82124,2.87642,0.0105901,False


Se muestran los resultados completos guardados de V1 hasta J9. Repetición desactivada: contextuales


**Decisión.** Se conserva global 93 y se desarrolla la pareja de 28 variables contextuales
más identidad: 29 entradas. Es una alternativa de entrenamiento que merece contraste, no una
afirmación de superioridad. El siguiente bloque comprueba cuánto pesa cada club dentro de la pareja.

<a id="equilibrio"></a>
## 5. Equilibrar el peso entre clubes

Un club con más registros puede dominar el ajuste conjunto. Se compara ponderar por registros
con asignar la mitad del peso a cada equipo, conservando recencia y afinidad dentro de él.
Este 50/50 entre clubes es distinto del promedio entre medias de la referencia. El tamaño efectivo
ESS describe concentración de pesos; no añade partidos ni garantiza independencia.

In [7]:
mostrar("Equilibrio y controles",comparar("equilibrio",["local_perfil_directos","xgb_poisson_sin_grupos","equipo_rival_previo"]))
mostrar("Soporte efectivo y concentración",tabla("equilibrio","resumen_soporte.csv"))
mostrar("Cambio pareado en desarrollo",pd.DataFrame([leer_json(INFORMES/"desarrollo/intervalos.json")["equilibrio"]]))
repetir_si_se_pide("equilibrio")

modelo,partidos_desarrollo,n_desarrollo,MAE_desarrollo,RMSE_desarrollo,sesgo_desarrollo,partidos_contraste,n_contraste,MAE_contraste,RMSE_contraste,sesgo_contraste,partidos_acumulado,n_acumulado,MAE_acumulado,RMSE_acumulado,sesgo_acumulado,hipotesis_retrospectiva
xgb_poisson_sin_grupos,59,118,2.99863,3.75474,0.156173,29,58,2.90109,3.73584,0.139625,88,176,2.96648,3.74852,0.15072,False
local_perfil_directos_equilibrado,59,118,2.97221,3.75629,0.374247,29,58,3.29398,4.26186,0.216865,88,176,3.07825,3.93009,0.322382,False
local_perfil_directos,59,118,3.02335,3.78079,0.330944,29,58,3.2923,4.20627,0.0896242,88,176,3.11198,3.9261,0.251418,False
equipo_rival_previo,59,118,3.03895,3.84346,0.396573,29,58,2.78058,3.58096,0.256943,88,176,2.9538,3.75898,0.350558,False


modelo,modo,n_propios,n_propios_actuales,ess_propios,masa_club,peso_maximo_dentro_club,fraccion_actual_dentro_club,afinidad_media,afinidad_minima
local_perfil_directos_equilibrado,igual_por_club,91.142,3.91477,15.4136,0.5,0.243121,0.562464,0.593469,0.363426


modelo,referencia,partidos,delta_RMSE,p025_descriptivo,p975_descriptivo,limite
local_perfil_directos_equilibrado,local_perfil_directos,59,-0.0244993,-0.106211,0.0549361,"Intervalo exploratorio por partidos, condicionado a datos y selección; no demuestra superioridad ni corrige búsqueda ni dependencia temporal."


Se muestran los resultados completos guardados de V1 hasta J9. Repetición desactivada: equilibrio


**Decisión.** La mejora en desarrollo es pequeña y no consistente en todos los periodos.
Ambas estrategias por pareja pasan a la rejilla; no se declara que equilibrar siempre sea mejor.

<a id="rejilla"></a>
## 6. Búsqueda de hiperparámetros: 144 configuraciones

Se fijan 48 combinaciones para global, 48 para pareja por registros y 48 para pareja equilibrada.
Cambian número de árboles, profundidad y regularización, manteniendo 55/25/13/7 y las entradas.
La selección usa el tramo de desarrollo; el ranking muestra también contraste y acumulado.
La mejor combinación de ese tramo no tiene por qué ser la mejor al ampliar la muestra.

In [8]:
rejilla=leer_json(RAIZ/"config/rejilla_historica.json")
mostrar("Las 144 configuraciones exactas",pd.DataFrame([dict(modelo=s["id"],estrategia=s["estrategia"],**s["candidato"]["parametros"]) for s in rejilla]))
assert len(rejilla)==144
mostrar("Ranking completo: selección por desarrollo",tabla("rejilla","ranking.csv"))
mostrar("Ganadores por estrategia y sus controles",tabla("rejilla","seleccion.csv"),abierto=True)
mostrar("Comparaciones pareadas de la rejilla",pd.DataFrame(leer_json(INFORMES/"desarrollo/intervalos.json")["rejilla"]))
finales=leer_json(RAIZ/"config/modelos.json")
mostrar("Configuraciones XGBoost conservadas",pd.DataFrame([dict(modelo=finales[k]["receta"]["id"],**finales[k]["receta"]["candidato"]["parametros"]) for k in ["global","pareja"]]))
repetir_si_se_pide("rejilla")

modelo,estrategia,objective,n_estimators,learning_rate,max_depth,min_child_weight,reg_lambda,reg_alpha,subsample,colsample_bytree,max_delta_step,tree_method,device,n_jobs,random_state,verbosity,validate_parameters
grid_global93_001,global93,count:poisson,100,0.03,2,5,10,0,1,0.8,0.7,hist,cpu,1,42,0,True
grid_global93_002,global93,count:poisson,100,0.03,2,5,20,0,1,0.8,0.7,hist,cpu,1,42,0,True
grid_global93_003,global93,count:poisson,200,0.03,2,5,10,0,1,0.8,0.7,hist,cpu,1,42,0,True
grid_global93_004,global93,count:poisson,200,0.03,2,5,20,0,1,0.8,0.7,hist,cpu,1,42,0,True
grid_global93_005,global93,count:poisson,100,0.03,2,10,10,0,1,0.8,0.7,hist,cpu,1,42,0,True
grid_global93_006,global93,count:poisson,100,0.03,2,10,20,0,1,0.8,0.7,hist,cpu,1,42,0,True
grid_global93_007,global93,count:poisson,200,0.03,2,10,10,0,1,0.8,0.7,hist,cpu,1,42,0,True
grid_global93_008,global93,count:poisson,200,0.03,2,10,20,0,1,0.8,0.7,hist,cpu,1,42,0,True
grid_global93_009,global93,count:poisson,100,0.03,3,5,10,0,1,0.8,0.7,hist,cpu,1,42,0,True
grid_global93_010,global93,count:poisson,100,0.03,3,5,20,0,1,0.8,0.7,hist,cpu,1,42,0,True


modelo,partidos_desarrollo,n_desarrollo,MAE_desarrollo,RMSE_desarrollo,sesgo_desarrollo,partidos_contraste,n_contraste,MAE_contraste,RMSE_contraste,sesgo_contraste,partidos_acumulado,n_acumulado,MAE_acumulado,RMSE_acumulado,sesgo_acumulado,estrategia,complejidad,hipotesis_retrospectiva
grid_pareja29_equilibrada_024,59,118,2.93797,3.72355,0.34962,29,58,3.22017,4.13303,0.144247,88,176,3.03097,3.86329,0.28194,pareja29_equilibrada,800,False
grid_pareja29_equilibrada_018,59,118,2.94346,3.72684,0.377727,29,58,3.12688,4.02922,0.27467,88,176,3.00391,3.82913,0.343765,pareja29_equilibrada,400,False
grid_pareja29_equilibrada_022,59,118,2.9381,3.72801,0.380692,29,58,3.12765,4.03106,0.276661,88,176,3.00057,3.83053,0.346409,pareja29_equilibrada,400,False
grid_pareja29_equilibrada_030,59,118,2.94435,3.73109,0.367931,29,58,3.29501,4.2608,0.209942,88,176,3.05991,3.91358,0.315866,pareja29_equilibrada,800,False
grid_pareja29_equilibrada_012,59,118,2.94489,3.73287,0.356612,29,58,3.27565,4.25497,0.196234,88,176,3.05389,3.91263,0.30376,pareja29_equilibrada,1600,False
grid_pareja29_equilibrada_020,59,118,2.95237,3.73711,0.362231,29,58,3.2265,4.13094,0.145957,88,176,3.04271,3.87132,0.290959,pareja29_equilibrada,800,False
grid_pareja29_equilibrada_016,59,118,2.95343,3.74096,0.343982,29,58,3.28784,4.26858,0.190203,88,176,3.06364,3.92268,0.293305,pareja29_equilibrada,1600,False
grid_pareja29_equilibrada_008,59,118,2.94245,3.74164,0.369845,29,58,3.15935,4.07201,0.274144,88,176,3.01393,3.85364,0.338307,pareja29_equilibrada,800,False
grid_pareja29_original_012,59,118,2.97869,3.74267,0.335912,29,58,3.33509,4.26465,0.0735267,88,176,3.09614,3.92237,0.249444,pareja29_original,1600,False
grid_pareja29_equilibrada_004,59,118,2.94704,3.74333,0.367115,29,58,3.1533,4.06459,0.270452,88,176,3.01501,3.85216,0.33526,pareja29_equilibrada,800,False


modelo,partidos_desarrollo,n_desarrollo,MAE_desarrollo,RMSE_desarrollo,sesgo_desarrollo,partidos_contraste,n_contraste,MAE_contraste,RMSE_contraste,sesgo_contraste,partidos_acumulado,n_acumulado,MAE_acumulado,RMSE_acumulado,sesgo_acumulado,estrategia,complejidad
grid_pareja29_equilibrada_024,59,118,2.93797,3.72355,0.34962,29,58,3.22017,4.13303,0.144247,88,176,3.03097,3.86329,0.28194,pareja29_equilibrada,800
grid_pareja29_original_012,59,118,2.97869,3.74267,0.335912,29,58,3.33509,4.26465,0.0735267,88,176,3.09614,3.92237,0.249444,pareja29_original,1600
grid_global93_030,59,118,2.99141,3.74799,0.142379,29,58,2.89177,3.73412,0.156769,88,176,2.95857,3.74342,0.147121,global93,800


modelo,control,delta_RMSE,intervalo_percentil95,advertencia
grid_pareja29_equilibrada_024,local_perfil_directos_equilibrado,-0.0327414,"[-0.15490396695883168, 0.0995725919774553]","Descriptivo, condicionado a selección; no corrige búsqueda ni dependencia temporal."
grid_pareja29_original_012,local_perfil_directos,-0.0381121,"[-0.09972432435035364, 0.02469128332163352]","Descriptivo, condicionado a selección; no corrige búsqueda ni dependencia temporal."
grid_global93_030,xgb_poisson_sin_grupos,-0.00675549,"[-0.03510188793710508, 0.02363693652345496]","Descriptivo, condicionado a selección; no corrige búsqueda ni dependencia temporal."


modelo,objective,n_estimators,learning_rate,max_depth,min_child_weight,reg_lambda,reg_alpha,subsample,colsample_bytree,max_delta_step,tree_method,device,n_jobs,random_state,verbosity,validate_parameters
grid_global93_030,count:poisson,100,0.05,3,10,20,0,1,0.8,0.7,hist,cpu,1,42,0,True
grid_pareja29_equilibrada_024,count:poisson,200,0.05,2,10,20,0,1,0.8,0.7,hist,cpu,1,42,0,True


Se muestran los resultados completos guardados de V1 hasta J9. Repetición desactivada: rejilla


**Decisión.** Se conservan `grid_global93_030`, `grid_pareja29_equilibrada_024` y equipo/rival.
Son las recetas operativas seleccionadas por el desarrollo; no se presentan como óptimos del
acumulado. Sus diferencias son pequeñas. Los intervalos usan partidos completos y son exploratorios:
5.000 réplicas en ajuste y 2.000 en equilibrio y rejilla, sin corregir toda la búsqueda.

<a id="confirmacion"></a>
## 7. Estabilidad en otra temporada

Se mantienen los contrastes de 17 ramas sobre los 380 partidos de 2025/26. Aportan otra perspectiva
temporal, sin dar a ese periodo más importancia solo por tener más registros. No todas las variantes
se evaluaron allí y no se oculta esa diferencia de cobertura.

In [9]:
mostrar("Plan de los 17 contrastes",pd.DataFrame([dict(familia=k,modelo=m) for k,v in leer_json(RAIZ/"config/confirmaciones_historicas.json").items() for m in v]))
mostrar("Temporada anterior: equipo, localía y total",tabla("confirmacion","metricas.csv"))
mostrar("Ajustes y convergencia",tabla("confirmacion","diagnostico_ajuste.csv"))
mostrar("Soporte de los ajustes por pareja",tabla("confirmacion","resumen_soporte.csv"))
repetir_si_se_pide("confirmacion")

familia,modelo
globales,poisson_glm
globales,xgb_cuadratico
globales,xgb_poisson
globales,xgb_poisson_moderada
globales,xgb_poisson_sin_grupos
referencias,equipo_rival_previo
referencias,liga_localia_previa
referencias,media_global_recencia
referencias,media_global_uniforme
reducidos,general_nucleo7_poisson


modelo,periodo,nivel,partidos,n,RMSE,MAE,sesgo,muestra
poisson_glm,2025/26,Equipo,380,760,3.92143,3.09847,-0.183841,Temporada anterior
poisson_glm,2025/26,Local,380,380,3.63835,2.87215,-0.189614,Temporada anterior
poisson_glm,2025/26,Visitante,380,380,4.18541,3.32479,-0.178068,Temporada anterior
poisson_glm,2025/26,Total partido,380,380,5.47823,4.29929,-0.367682,Temporada anterior
xgb_cuadratico,2025/26,Equipo,380,760,3.71725,2.96589,-0.172305,Temporada anterior
xgb_cuadratico,2025/26,Local,380,380,3.42446,2.75012,0.0317971,Temporada anterior
xgb_cuadratico,2025/26,Visitante,380,380,3.9886,3.18166,-0.376406,Temporada anterior
xgb_cuadratico,2025/26,Total partido,380,380,5.0749,4.09826,-0.344609,Temporada anterior
xgb_poisson,2025/26,Equipo,380,760,3.73798,2.99184,-0.191643,Temporada anterior
xgb_poisson,2025/26,Local,380,380,3.4465,2.79379,0.0273268,Temporada anterior


modelo,observaciones_train,partidos_train,observaciones_valid,variables_salida,n_columnas_valor_excluido,peso_medio,peso_minimo,peso_maximo,RMSE_train,RMSE_train_ponderado,MAE_train,n_iter,correcciones_dominio_valid,segundos_ajuste_y_prediccion,columnas_entrada,segundos,ajustes,ajustes_con_aviso
poisson_glm,1879.33,939.665,2.17765,277.851,6.03438,1,0.0898056,10.6635,3.7201,3.32919,2.92454,149.327,0,0.771334,Sin datos,Sin datos,349,0
xgb_cuadratico,1879.33,939.665,2.17765,277.851,6.03438,1,0.0898056,10.6635,3.4621,2.88411,2.69752,100,0,1.58586,Sin datos,Sin datos,349,0
xgb_poisson,1879.33,939.665,2.17765,277.851,6.03438,1,0.0898056,10.6635,3.57921,3.12361,2.80467,100,0,1.6763,Sin datos,Sin datos,349,0
xgb_poisson_moderada,1879.33,939.665,2.17765,277.851,6.03438,1,0.139357,9.29223,3.56467,3.1833,2.79567,100,0,1.67204,Sin datos,Sin datos,349,0
xgb_poisson_sin_grupos,1879.33,939.665,2.17765,232.851,6.03438,1,0.0898056,10.6635,3.58809,3.13263,2.80723,100,0,1.44123,Sin datos,Sin datos,349,0
general_nucleo7_poisson,1879.33,939.665,Sin datos,14,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,7,0.172643,349,0
pareja_analogos,162.4,159.205,Sin datos,0,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,13,0.093696,380,0
global_perfil_localia,1879.33,939.665,Sin datos,45,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,18,0.442129,349,0
local_perfil_directos,162.4,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,29,0.38849,380,0
local_perfil_directos_equilibrado,162.4,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,Sin datos,29,Sin datos,380,0


modelo,modo,n_propios,n_propios_actuales,ess_propios,masa_club,peso_maximo_dentro_club,fraccion_actual_dentro_club,afinidad_media,afinidad_minima
local_perfil_directos_equilibrado,igual_por_club,81.2,18.5,32.1013,0.5,0.111124,0.640191,0.650945,0.330688
grid_pareja29_equilibrada_024,igual_por_club,81.2,18.5,32.1013,0.5,0.111124,0.640191,0.650945,0.330688
grid_pareja29_original_012,por_registros,81.2,18.5,32.1013,0.5,0.111124,0.640191,0.650945,0.330688


Se muestran los resultados completos guardados de V1 hasta J9. Repetición desactivada: confirmacion


<a id="cierre-desarrollo"></a>
## 8. Qué llevamos al contraste

Las 38 variantes abarcan referencias, algoritmos, bloques de entrada y formas de ponderar; no son
38 algoritmos distintos. La rejilla añade 144 configuraciones. El 03 compara las tres recetas,
comprueba nueve repartos de temporada y analiza errores. El estudio documenta un proceso retrospectivo;
el seguimiento de partidos futuros aportará una comprobación más independiente.
[docs/inventario_pruebas.md](../docs/inventario_pruebas.md) permite localizar cada prueba.